# 03 — Trovare le testate "parenti"

Due testate sono **parenti** se pubblicano lo **stesso testo**: articoli copiati, comunicati ripresi, pezzi generati
dallo stesso modello. Se due parenti finiscono una nel training e una nel test, il classificatore può riconoscere
il testo della sorella invece di giudicare l'affidabilità.

Regole di questo notebook, per renderlo inattaccabile:

- **cieco alle etichette:** i livelli NewsGuard vengono tolti appena caricato il dataset e non si usano mai;
- **automatico:** nessun articolo letto a mano, nessun LLM;
- **verificato con un dato esterno:** il proprietario dei siti riportato da NewsGuard, usato **solo alla fine** per controllare i gruppi.

In [1]:
import re
import numpy as np
import pandas as pd
import networkx as nx
from datasketch import MinHash, MinHashLSH
from comune import carica_dataset, NEWSGUARD, DERIVED, RESULTS

df = carica_dataset().drop(columns=["label", "score"])   # niente etichette in questo notebook
print(len(df), "articoli,", df.domain.nunique(), "testate")

19476 articoli, 439 testate


## 1. Normalizzare il testo

Per confrontare gli articoli togliamo le differenze che non contano: maiuscole, punteggiatura e **numeri**
(sostituiti da `#`, così due articoli-modello con statistiche diverse risultano uguali).

In [2]:
def normalizza(testo):
    testo = re.sub(r"\d+", "#", str(testo).lower())
    testo = re.sub(r"[^a-z# ]", " ", testo)
    return re.sub(r"\s+", " ", testo).strip()

print(normalizza("In 2024, Cook County collected $12.5 million in taxes!"))
df["norm"] = df.text.map(normalizza)

in # cook county collected # # million in taxes


## 2. Articoli identici su testate diverse

Il caso più semplice: lo stesso identico testo (dopo la normalizzazione) pubblicato da almeno due testate.

In [3]:
testate_per_testo = df.groupby("norm").domain.nunique()
identici = df[df.norm.map(testate_per_testo) >= 2]
print("articoli con un testo identico su un'altra testata:", len(identici))
print("testate coinvolte:", identici.domain.nunique())

articoli con un testo identico su un'altra testata: 1557
testate coinvolte: 151


## 3. Articoli quasi uguali (MinHash)

Gli articoli copiati spesso non sono identici: cambia il nome della città, un paragrafo in fondo, la firma.
Per trovarli:

1. ogni articolo viene spezzato in **pezzetti di 5 parole consecutive**;
2. due articoli sono **quasi uguali** se condividono almeno metà dei pezzetti
   (somiglianza di Jaccard $\geq 0{,}5$: pezzetti in comune diviso pezzetti totali dei due articoli);
3. confrontare tutte le coppie (quasi 200 milioni) sarebbe lentissimo: **MinHash** riassume ogni articolo in
   un'impronta di 128 numeri e **LSH** propone solo le coppie candidate. È la tecnica standard per trovare
   i duplicati nei dati di addestramento dei modelli linguistici.

Teniamo solo le coppie di articoli di **testate diverse**.

In [4]:
def pezzetti(testo_norm, n=5):
    parole = testo_norm.split()
    return {" ".join(parole[i:i + n]) for i in range(max(1, len(parole) - n + 1))}

lsh = MinHashLSH(threshold=0.5, num_perm=128)
impronte = []
for i, testo in enumerate(df.norm):
    m = MinHash(num_perm=128, seed=1)
    m.update_batch([p.encode() for p in pezzetti(testo)])
    impronte.append(m)
    lsh.insert(i, m)

testata = df.domain.values
coppie = []
for i, m in enumerate(impronte):
    for j in lsh.query(m):
        if j > i and testata[i] != testata[j]:
            jaccard = m.jaccard(impronte[j])
            if jaccard >= 0.5:
                coppie.append((i, j, round(jaccard, 3)))
coppie = pd.DataFrame(coppie, columns=["articolo_a", "articolo_b", "jaccard"])
coppie["testata_a"], coppie["testata_b"] = testata[coppie.articolo_a], testata[coppie.articolo_b]

print("coppie di articoli quasi uguali su testate diverse:", len(coppie))
quasi = set(coppie.articolo_a) | set(coppie.articolo_b)
print("articoli che hanno almeno una copia su un'altra testata:", len(quasi), f"({len(quasi) / len(df):.0%})")

coppie di articoli quasi uguali su testate diverse: 1451430
articoli che hanno almeno una copia su un'altra testata: 4740 (24%)


Un esempio di coppia **quasi uguale ma non identica** (somiglianza tra 0,6 e 0,8):

In [5]:
es = coppie[(coppie.jaccard > 0.6) & (coppie.jaccard < 0.8)].iloc[0]
for lato in ["a", "b"]:
    print(f"--- {es['testata_' + lato]} ---")
    print(df.text[es["articolo_" + lato]][:500], "...\n")
print("somiglianza:", es.jaccard)

--- nemontananews.com ---
Federal Energy Regulatory Commission (FERC) Chairman Willie L. Phillips | www.ferc.gov

Federal Energy Regulatory Commission (FERC) Chairman Willie L. Phillips | www.ferc.gov

There was a single gas station in cities across Phillips County selling regular gas at the price of $3.09 per gallon in the week ending Mar. 15, according to customer reports submitted to GasBuddy.com

Of the six reported gas stations in cities within Phillips County, just one offers regular gas.

The average reported pric ...

--- pensacolatimes.com ---
Federal Energy Regulatory Commission (FERC) Commissioner Allison Clements | www.ferc.gov

Federal Energy Regulatory Commission (FERC) Commissioner Allison Clements | www.ferc.gov

There was a single gas station in cities across Escambia County selling regular gas at the lowest price of $2.65 per gallon in the week ending Mar. 15, according to customer reports submitted to GasBuddy.com

The average reported cost of regular gas in cities w

## 4. Dai legami tra articoli ai gruppi di testate

- Due testate sono **collegate** se hanno **almeno 2 articoli** quasi uguali in comune.
  Una sola copia può essere una notizia ripresa una volta; due o più indicano un rapporto stabile.
- Un **gruppo** è un insieme di testate collegate anche **indirettamente** (A–B e B–C: A, B e C stanno insieme).
- Una testata senza legami è **indipendente** (gruppo da sola).

In [6]:
articoli_in_comune = coppie.groupby(["testata_a", "testata_b"]).size()

def gruppi(min_articoli=2, min_jaccard=0.5):
    c = coppie[coppie.jaccard >= min_jaccard].groupby(["testata_a", "testata_b"]).size()
    G = nx.Graph()
    G.add_nodes_from(df.domain.unique())
    G.add_edges_from(c[c >= min_articoli].index)
    componenti = sorted(nx.connected_components(G), key=len, reverse=True)
    return pd.Series({t: k for k, comp in enumerate(componenti) for t in comp}, name="gruppo")

gruppo = gruppi()
dimensione = gruppo.map(gruppo.value_counts())
print("gruppi con almeno 2 testate:", (gruppo.value_counts() >= 2).sum())
print("testate in un gruppo:", (dimensione >= 2).sum(), "su", len(gruppo))
print("articoli di testate in un gruppo:", df.domain.map(dimensione).ge(2).sum(), "su", len(df))
print()
riassunto = pd.DataFrame({"testate": gruppo.value_counts()})
riassunto["esempi"] = [", ".join(sorted(gruppo[gruppo == g].index)[:3]) for g in riassunto.index]
riassunto[riassunto.testate >= 2].head(12)

gruppi con almeno 2 testate: 10
testate in un gruppo: 236 su 439
articoli di testate in un gruppo: 9807 su 19476



,testate,esempi
gruppo,,
0,131,"aroostooknews.com, boisecitywire.com, cantonre..."
1,42,"alabamanow.com, albertleatribune.com, american..."
2,31,"americanlookout.com, amgreatness.com, christia..."
3,8,"data.huffingtonpost.com, editorial.huffingtonp..."
4,7,"cardinalpine.com, coppercourier.com, gandernew..."
5,4,"clantonadvertiser.com, salisburypost.com, stat..."
6,4,"gopdailybrief.com, patriotjournal.org, politic..."
7,4,"thewildcard.com, w-j.co, westernjournal.com"
8,3,"americustimesrecorder.com, brewtonstandard.com..."


## 5. Le scelte cambiano i gruppi?

Le due soglie (almeno 2 articoli in comune, somiglianza almeno 0,5) sono state fissate prima.
Qui le facciamo variare per vedere quanto cambiano i gruppi.

In [7]:
righe = []
for min_jaccard in [0.5, 0.7, 0.9]:
    for min_articoli in [1, 2, 3, 5]:
        g = gruppi(min_articoli, min_jaccard)
        dim = g.value_counts()
        righe.append({"somiglianza min.": min_jaccard, "articoli in comune min.": min_articoli,
                      "gruppi (≥2 testate)": (dim >= 2).sum(), "testate in un gruppo": dim[dim >= 2].sum(),
                      "gruppo più grande": dim.iloc[0]})
sensibilita = pd.DataFrame(righe)
sensibilita

,somiglianza min.,articoli in comune min.,gruppi (≥2 testate),testate in un gruppo,gruppo più grande
0,0.5,1,7,264,131
1,0.5,2,10,236,131
2,0.5,3,13,215,131
3,0.5,5,20,205,131
4,0.7,1,9,259,131
5,0.7,2,11,234,131
6,0.7,3,14,213,128
7,0.7,5,21,199,123
8,0.9,1,14,240,113
9,0.9,2,14,201,100


## 6. Verifica con un dato esterno: il proprietario

NewsGuard riporta per ogni sito il **proprietario** (campo `Owner`). Non l'abbiamo usato per costruire i gruppi:
ora serve solo a controllarli. Per ogni testata guardiamo:

- **precisione:** tra le testate del suo gruppo, quante hanno davvero lo stesso proprietario;
- **richiamo:** tra le testate con lo stesso proprietario, quante sono finite nel suo gruppo.

Poi facciamo la media su tutte le testate (misura *B-cubed*: ogni testata pesa uguale, il gruppo più grande non domina).
Le testate senza proprietario noto contano come proprietario a sé.

In [8]:
ng = pd.read_csv(NEWSGUARD, low_memory=False).drop_duplicates("Domain").set_index("Domain")
proprietario = pd.Series({t: (ng.Owner.get(t) if isinstance(ng.Owner.get(t), str) and ng.Owner.get(t) != "Unknown" else "?" + t)
                          for t in gruppo.index})
print("testate che condividono il proprietario con almeno un'altra:", (proprietario.map(proprietario.value_counts()) >= 2).sum())

def precisione_richiamo(gruppo):
    p, r = [], []
    for t in gruppo.index:
        stesso_gruppo = set(gruppo.index[gruppo == gruppo[t]])
        stesso_proprietario = set(proprietario.index[proprietario == proprietario[t]])
        comuni = len(stesso_gruppo & stesso_proprietario)
        p.append(comuni / len(stesso_gruppo))
        r.append(comuni / len(stesso_proprietario))
    return np.mean(p), np.mean(r)

for min_jaccard in [0.5, 0.7, 0.9]:
    for min_articoli in [1, 2, 3, 5]:
        p, r = precisione_richiamo(gruppi(min_articoli, min_jaccard))
        sensibilita.loc[(sensibilita["somiglianza min."] == min_jaccard) & (sensibilita["articoli in comune min."] == min_articoli),
                        ["precisione", "richiamo"]] = round(p, 3), round(r, 3)
sensibilita

testate che condividono il proprietario con almeno un'altra: 254


,somiglianza min.,articoli in comune min.,gruppi (≥2 testate),testate in un gruppo,gruppo più grande,precisione,richiamo
0,0.5,1,7,264,131,0.789,0.906
1,0.5,2,10,236,131,0.926,0.874
2,0.5,3,13,215,131,0.961,0.801
3,0.5,5,20,205,131,0.984,0.772
4,0.7,1,9,259,131,0.870,0.885
5,0.7,2,11,234,131,0.931,0.854
6,0.7,3,14,213,128,0.963,0.783
7,0.7,5,21,199,123,0.984,0.738
8,0.9,1,14,240,113,0.899,0.807
9,0.9,2,14,201,100,0.945,0.726


Quali proprietari **non** vengono riuniti in un solo gruppo? Sono siti dello stesso proprietario che non condividono testo:
per il classificatore sono testate diverse.

In [9]:
righe = []
for prop, testate in proprietario.groupby(proprietario):
    if len(testate) >= 2 and not prop.startswith("?"):
        righe.append({"proprietario": prop, "testate": len(testate), "gruppi in cui finiscono": gruppo[testate.index].nunique()})
pd.DataFrame(righe).sort_values("testate", ascending=False)

,proprietario,testate,gruppi in cui finiscono
8,Metric Media LLC,138,8
1,"Boone Newspapers, Inc.",56,10
13,"Righters Group, LLC",9,9
2,BuzzFeed and Verizon Media Group/Verizon Commu...,8,1
7,"Liftable Media, Inc.",6,3
15,Salem Media Group,5,2
17,"Star News Digital Media, Inc.",5,1
14,Ryan Mauldin,4,1
4,Courier Newsroom Inc.,4,1
6,Good Information Inc.,3,1


## 7. Un secondo segnale: paragrafi condivisi

Oltre agli articoli copiati, alcune testate condividono **paragrafi fissi** (didascalie, note, appelli, lettori audio).
Contiamo i paragrafi di almeno 8 parole che compaiono in più testate.
Questo segnale **non** lo usiamo per formare i gruppi: frasi di servizio generiche
(es. inviti alla newsletter) collegano anche testate che non hanno niente in comune.
Lo useremo invece per **ripulire il testo** (notebook successivi).

In [10]:
paragrafi = df[["article_id", "domain", "text"]].assign(paragrafo=df.text.astype(str).str.split("\n")).explode("paragrafo")
paragrafi["paragrafo"] = paragrafi.paragrafo.map(normalizza)
paragrafi = paragrafi[paragrafi.paragrafo.str.split().str.len() >= 8].drop_duplicates(["article_id", "paragrafo"])
testate_per_paragrafo = paragrafi.groupby("paragrafo").domain.nunique()
paragrafi["condiviso"] = paragrafi.paragrafo.map(testate_per_paragrafo) >= 2

print("paragrafi (≥ 8 parole):", len(paragrafi))
print("paragrafi presenti in almeno 2 testate:", paragrafi.condiviso.sum(), f"({paragrafi.condiviso.mean():.0%})")
print("articoli con almeno un paragrafo condiviso:", paragrafi[paragrafi.condiviso].article_id.nunique())
print()
print("I paragrafi presenti nel maggior numero di testate:")
testate_per_paragrafo.sort_values(ascending=False).head(10).rename(lambda s: s[:110])

paragrafi (≥ 8 parole): 297751
paragrafi presenti in almeno 2 testate: 49105 (16%)
articoli con almeno un paragrafo condiviso: 5802

I paragrafi presenti nel maggior numero di testate:


paragrafo
the data in this article comes from the h #b employer data hub provided by the u s citizenship and immigration    107
employers are classified under industries based on their reported operations and may be involved in multiple i    107
the h #b visa program allows u s employers to temporarily hire foreign workers in specialty occupations requir    107
public opinion on immigration varies with # of americans favoring maintaining current legal immigration levels    107
the united states admits around # million legal immigrants annually most of whom receive permanent residence g    107
drivers in the united states have experienced a rollercoaster of gas prices in recent years due to supply disr    105
gas data refers to the price of gas when paid for by cash not credit gas stations can charge drivers more for     105
federal energy regulatory commission ferc commissioner mark c christie www ferc gov                               105
according to the u s energy information admini

## 8. Salvataggio

- `Results/03_gruppi_testate.csv`: per ogni testata il gruppo e la sua dimensione (nessuna etichetta);
- `Dataset/derived/`: le coppie di articoli quasi uguali e i paragrafi condivisi, per i notebook successivi.

In [11]:
pd.DataFrame({"gruppo": gruppo, "testate_nel_gruppo": dimensione}).rename_axis("domain").to_csv(RESULTS + "03_gruppi_testate.csv")
coppie.to_csv(DERIVED + "03_coppie_quasi_uguali.csv.gz", index=False)
paragrafi[paragrafi.condiviso][["article_id", "domain", "paragrafo"]].to_csv(DERIVED + "03_paragrafi_condivisi.csv.gz", index=False)
print("Salvato.")

Salvato.
